# Data preprocessing: normalising `description` and `unit_of_measure`, feature matrix and PCA

Rule-based (regex) normalisation of the supplier description into structured
fields: pack count, per-item size (grams or ml), product dimensions and storage
temperature. Plus price-per-normalised-unit using `unit_of_measure`.

Everything is a pure function of one row, so it can be applied to train and test
with no fitted state (no leakage).

Why regex and not only text features: `unit_of_measure` is a weak signal (CS, CTN and
PK are used interchangeably), and only ~53% of descriptions contain a size. The
parser is one layer; a character n-gram TF-IDF on the cleaned text is the next.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

DATA = Path("..") / "data"  # notebook lives in notebooks/
train = pd.read_csv(DATA / "train.csv")
print(train.shape)

## 1. Unit table

Every spelling maps to a dimension and a multiplier to the base unit
(grams for mass, millilitres for volume). Length (cm, mm) is handled separately.

In [ ]:
UNIT_TABLE: dict[str, tuple[str, float]] = {
    "kgs": ("mass", 1000.0), "kg": ("mass", 1000.0),
    "gms": ("mass", 1.0), "gm": ("mass", 1.0), "grm": ("mass", 1.0),
    "gr": ("mass", 1.0), "g": ("mass", 1.0),
    "mg": ("mass", 0.001),
    "litres": ("volume", 1000.0), "liters": ("volume", 1000.0),
    "litre": ("volume", 1000.0), "liter": ("volume", 1000.0),
    "ltr": ("volume", 1000.0), "lt": ("volume", 1000.0), "l": ("volume", 1000.0),
    "cl": ("volume", 10.0), "ml": ("volume", 1.0),
}
# Longest first so "kgs" wins over "kg", "kg" over "g", etc.
_UNITS = "|".join(sorted(UNIT_TABLE, key=len, reverse=True))
_NUM = r"\d+(?:\.\d+)?"

## 2. Regex patterns

Order matters. The dimension pattern runs first (`46X33CM` would otherwise read as
pack 46 x 33 cm), then the size-with-pack patterns, then the bare size and the
pack-only patterns.

In [ ]:
# Supplier housekeeping tokens with no product information
_NOISE = [
    re.compile(r"\*+\s*(?:sub|use)\b[^*]*\*+"),  # *SUB 44847*, *** USE ITEM 5156 ***
    re.compile(r"\b(?:item|plu|line)\s*#?\s*\d+\b"),  # ITEM 16405, PLU 91549, LINE 74022
    re.compile(r"ref\s*#\s*[a-z]?\d+"),  # REF#A15317 (no \b: "6/CSREF#A85000" has no space)
    re.compile(r"\brev\s*\d+\b"),  # REV2
    re.compile(r"\(\s*sub\s*\)|\bsub\b"),  # (SUB)
    re.compile(r"\bitem\b"),  # bare ITEM left over after its number was removed
    re.compile(r"\*+"),  # stray asterisks
]

_DIM_PAIR = re.compile(rf"(?P<a>{_NUM})\s*x\s*/?\s*(?P<b>{_NUM})\s*/?\s*(?P<u>cm|mm)(?![a-z])")
_DIM_ONE = re.compile(rf"(?P<a>{_NUM})\s*(?P<u>cm|mm)(?![a-z])")

# 6/1KG, 24 x 330ML, 12/750-ml, 20x300 g, 6X2KG  (pack first)
_PACK_X_SIZE = re.compile(
    rf"(?P<pack>\d+)\s*(?:x|/|of)\s*(?P<size>{_NUM})\s*-?\s*(?P<unit>{_UNITS})(?![a-z])"
)
# 110g x24, 1kg/6  (size first)
_SIZE_X_PACK = re.compile(
    rf"(?P<size>{_NUM})\s*-?\s*(?P<unit>{_UNITS})(?![a-z])\s*(?:x|/)\s*(?P<pack>\d+)(?![\d.])"
)
# 1KG, 750mL, 1.5 ltr, 25/KG
_SIZE = re.compile(rf"(?P<size>{_NUM})\s*[-/]?\s*(?P<unit>{_UNITS})(?![a-z])")

# Pack-only patterns, tried in order when no size-with-pack pattern matched
_PACK_DOZ = re.compile(r"(?P<pack>\d+)\s*/?\s*(?:dz|doz|doze|dozen)(?![a-z])")
_PACK_EACH = re.compile(r"(?P<pack>\d+)\s*[-/]?\s*(?:ea|each|pcs?|pieces?|pc|ct|count)(?![a-z])")
# "9/1", "12/1": N items of one unit each (same convention as "6/1KG"); unit size unknown
_PACK_N_OVER_ONE = re.compile(r"(?P<pack>\d+)\s*/\s*1(?![\d.a-z])")
_PACK_PER = re.compile(
    r"(?P<pack>\d+)\s*(?:per\s*|/\s*)(?:case|cs|ctn|carton|box|pk|pack|bag)(?![a-z])"
)
_PACK_CASE_OF = re.compile(r"case\s*of\s*(?P<pack>\d+)")  # "case of 500", "caseof 100"
# CTN100, PK250, BOX-6, FLOURPK10 (keyword glued to a word is split in _split_glued)
_PACK_PREFIX = re.compile(
    r"(?:(?<![a-z])(?:ctn|cs|case|pkt|pack|box|bx|bag)|pk)[\s/-]*(?P<pack>\d+)"
)
# GoldLeaf x10: last resort, a bare x<n> that no size pattern claimed
_PACK_TIMES = re.compile(r"(?<![\d.])x\s*(?P<pack>\d+)(?![\d.])")  # "thigh x 6" after glue split
# "Ale AMB 24x", "Pantry Pro 10X": trailing x with nothing after it
_PACK_TRAILING_X = re.compile(r"(?P<pack>\d+)\s*x\s*$")
# "Croissant 48/", "48/ PROMO": a trailing slash after the count
_PACK_TRAILING_SLASH = re.compile(r"(?P<pack>\d+)\s*/\s*(?=(?:promo|new|disc|spec|imp|nsf)\b|$)")
# Croissant 60G (48), Ale (24 btl), Ale (24 (truncated)
_PACK_PAREN = re.compile(r"\(\s*(?P<pack>\d+)\s*(?:btl|bt|\)|$)")

# unit glued to a following keyword: "210mlpk6" -> "210 ml pk6", "400 mlx2" -> "400 ml x2",
# "1 KGDISC" -> "1 KG DISC"
_UNIT_BEFORE_PACK_KW = re.compile(
    rf"\b({_UNITS})(?=(?:pk|pkt|ctn|cs|case|box|bx|bag|pack|pcs?|ea|btl|x|new|promo|disc|spec|imp|nsf|amb|frz|frozen|chilled|refrigerated|dry|shelf)(?![a-z]))"
)

_STORAGE_FROZEN = re.compile(r"\b(?:frozen|frz|iqf)\b")
_STORAGE_CHILLED = re.compile(r"\b(?:chilled|chill|chiller|refrigerated|refrig|ref)\b")
_STORAGE_AMBIENT = re.compile(r"\b(?:ambient|amb|shelf stable|shelfstable|dry store)\b")

## 3. Cleaning and parsing functions

In [ ]:
def _clean(raw: str) -> str:
    s = str(raw).lower().replace("×", "x").replace("’", "'")
    for pattern in _NOISE:
        s = pattern.sub(" ", s)
    return s


def _split_glued(s: str) -> str:
    """'ctn100' -> 'ctn 100', '1kg' -> '1 kg', '2kgrefrigerated' -> '2 kg refrigerated'."""
    s = re.sub(r"(?<=\d)(?=[a-z])|(?<=[a-z])(?=\d)", " ", s)
    s = re.sub(r"(?<=[a-z])(?=(?:refrigerated|chilled|ambient|frozen|shelf\s?stable)\b)", " ", s)
    s = re.sub(r"(?<=[a-z])(?=(?:ctn|pk)(?![a-z]))", " ", s)  # "dry storectn12", "flourpk10"
    s = _UNIT_BEFORE_PACK_KW.sub(r"\1 ", s)
    return re.sub(r"\s+", " ", s).strip()


def normalise_text(raw: str) -> str:
    """Lowercase, strip noise tokens and split glued numbers/words. Use for text features."""
    return _split_glued(_clean(raw))


def _extract(pattern: re.Pattern, s: str) -> tuple[re.Match | None, str]:
    """Return the first match and the string with that span blanked out."""
    m = pattern.search(s)
    if m is None:
        return None, s
    return m, s[: m.start()] + " " + s[m.end():]


def _to_base(number: str, unit: str) -> tuple[float, str]:
    dimension, factor = UNIT_TABLE[unit]
    return float(number) * factor, dimension


def _size_text(out: dict) -> str | None:
    """Human-readable per-item size for the `size` attribute.

    Mass or volume wins when both are present: '1 kg', '2.5 kg', '750 ml', '1.5 L'.
    Otherwise the product dimensions in mm: '200 mm', '240 x 240 mm'.
    """
    if out["size_kind"] == "mass" or out["size_kind"] == "volume":
        qty = out["size_qty"]
        if out["size_kind"] == "mass":
            return f"{qty / 1000:g} kg" if qty >= 1000 else f"{qty:g} g"
        return f"{qty / 1000:g} L" if qty >= 1000 else f"{qty:g} ml"
    if not np.isnan(out["dim_a_mm"]):
        a = f"{out['dim_a_mm']:g}"
        if np.isnan(out["dim_b_mm"]):
            return f"{a} mm"
        return f"{a} x {out['dim_b_mm']:g} mm"
    return None


def parse_description(raw: str) -> dict:
    """Extract pack count, per-item size, dimensions and storage from one description.

    pack_qty   : inner items per selling unit (NaN if not stated)
    size_qty   : per-item content in grams (mass) or ml (volume) (NaN if not stated)
    size_kind  : 'mass' | 'volume' | None
    dim_a_mm, dim_b_mm : product dimensions in mm (NaN if not stated)
    storage    : 'frozen' | 'chilled' | 'ambient' | None
    size       : readable per-item size for the attribute ('1 kg', '240 x 240 mm'), or None
    """
    base = normalise_text(raw)
    out: dict = {
        "pack_qty": np.nan,
        "size_qty": np.nan,
        "size_kind": None,
        "dim_a_mm": np.nan,
        "dim_b_mm": np.nan,
        "storage": None,
        "size": None,
    }

    # storage is read from the whole text, before any span is removed
    if _STORAGE_FROZEN.search(base):
        out["storage"] = "frozen"
    elif _STORAGE_CHILLED.search(base):
        out["storage"] = "chilled"
    elif _STORAGE_AMBIENT.search(base):
        out["storage"] = "ambient"

    s = base

    m, s = _extract(_DIM_PAIR, s)
    if m is not None:
        factor = 10.0 if m.group("u") == "cm" else 1.0
        out["dim_a_mm"] = float(m.group("a")) * factor
        out["dim_b_mm"] = float(m.group("b")) * factor
    else:
        m, s = _extract(_DIM_ONE, s)
        if m is not None:
            factor = 10.0 if m.group("u") == "cm" else 1.0
            out["dim_a_mm"] = float(m.group("a")) * factor

    # size and pack together, most specific pattern first
    m, s = _extract(_PACK_X_SIZE, s)
    if m is not None:
        out["pack_qty"] = float(m.group("pack"))
        out["size_qty"], out["size_kind"] = _to_base(m.group("size"), m.group("unit"))
    else:
        m, s = _extract(_SIZE_X_PACK, s)
        if m is not None:
            out["pack_qty"] = float(m.group("pack"))
            out["size_qty"], out["size_kind"] = _to_base(m.group("size"), m.group("unit"))
        else:
            m, s = _extract(_SIZE, s)
            if m is not None:
                out["size_qty"], out["size_kind"] = _to_base(m.group("size"), m.group("unit"))

    if np.isnan(out["pack_qty"]):
        for pattern, multiplier in (
            (_PACK_DOZ, 12.0),
            (_PACK_EACH, 1.0),
            (_PACK_N_OVER_ONE, 1.0),
            (_PACK_PER, 1.0),
            (_PACK_CASE_OF, 1.0),
            (_PACK_PREFIX, 1.0),
            (_PACK_PAREN, 1.0),
            (_PACK_TIMES, 1.0),
            (_PACK_TRAILING_X, 1.0),
            (_PACK_TRAILING_SLASH, 1.0),
        ):
            m, s = _extract(pattern, s)
            if m is not None:
                out["pack_qty"] = float(m.group("pack")) * multiplier
                break

    out["size"] = _size_text(out)
    # what is left once the parsed spans are blanked out (input for the text ablation)
    out["residual_text"] = " ".join(s.split())
    return out


def parse_frame(df: pd.DataFrame, description_col: str = "description") -> pd.DataFrame:
    """Parse every description in a DataFrame. Returns parsed columns aligned to df.index."""
    parsed = pd.DataFrame([parse_description(d) for d in df[description_col]], index=df.index)
    parsed["size_kind"] = parsed["size_kind"].astype("object")
    parsed["storage"] = parsed["storage"].astype("object")
    parsed["has_pack"] = parsed["pack_qty"].notna().astype(int)
    parsed["has_size"] = parsed["size_qty"].notna().astype(int)
    parsed["has_dims"] = parsed["dim_a_mm"].notna().astype(int)
    return parsed

## 4. Check the parser on known examples

Each row is a description I checked by hand against the expected reading. The
assertions fail loudly if a pattern change breaks one.

In [ ]:
EXPECTED = [
    # raw description, pack_qty, size_qty, size_kind, storage
    ("meat 6/1KG LINE 52985", 6, 1000, "mass", None),
    ("Harbour Select plastic cup with lid470mL 1000 per case REV2", 1000, 470, "volume", None),
    ("Pantry-Pro Wine 12/750-ml REF#A15317", 12, 750, "volume", None),
    ("Focacciia20x300 g", 20, 300, "mass", None),
    ("METRO VALUE, DETERGENT, 20 LITRE, BOX 2", 2, 20000, "volume", None),
    ("YGHURT GOLDLEAF 6X2KGREFRIGERATED", 6, 2000, "mass", "chilled"),
    ("ITEM 88639 Metro Value Beer AMB 24/375mL REV2", 24, 375, "volume", "ambient"),
    ("Metro Value Croissant 60G (48)", 48, 60, "mass", None),
    ("BEER GLASS HARBOUR SELECT 210MLPK6", 6, 210, "volume", None),
    ("FLOURPK10", 10, np.nan, None, None),
    ("SUGAR 25 KG", np.nan, 25000, "mass", None),
    ("BASMATI RICE 25/KG", np.nan, 25000, "mass", None),
    ("IMP Chef's Choice Muffin 110g x24", 24, 110, "mass", None),
    ("Water 24 x 330ML", 24, 330, "volume", None),
    ("Egg keep chilled 30X1KG", 30, 1000, "mass", "chilled"),
    ("Royal Table hash browns 2.5Kg frozen", np.nan, 2500, "mass", "frozen"),
    ("pantry pro case of 1000 ITEM77550", 1000, np.nan, None, None),
    ("GoldLeaf x10", 10, np.nan, None, None),
    ("HAND CREAM, 30 G,PK 250", 250, 30, "mass", None),
]

def _same(a, b) -> bool:
    if a is None or (isinstance(a, float) and np.isnan(a)):
        return b is None or (isinstance(b, float) and np.isnan(b))
    return a == b

failures = []
for raw, pack, size, kind, storage in EXPECTED:
    got = parse_description(raw)
    ok = (
        (np.isnan(pack) and np.isnan(got["pack_qty"]) or got["pack_qty"] == pack)
        and (np.isnan(size) and np.isnan(got["size_qty"]) or got["size_qty"] == size)
        and _same(kind, got["size_kind"])
        and _same(storage, got["storage"])
    )
    if not ok:
        failures.append((raw, got))

# dimensions: 46X33CM -> 460 x 330 mm, and the pack still parses
got = parse_description("GASTRONORM PAN, CHEF'S CHOICE, 46X33CM, 12 EA/CS")
assert got["dim_a_mm"] == 460 and got["dim_b_mm"] == 330 and got["pack_qty"] == 12, got

print("failures:", len(failures))
for raw, got in failures:
    print(raw, "->", got)
assert not failures, "some examples parse differently from the hand-checked reading"
print("all", len(EXPECTED) + 1, "hand-checked examples pass")

### Regression cases from review

Rows from a manual review of the residual list. Each case checks only the fields it lists.

In [ ]:
REVIEW_CASES = [
    ("PANTRY/PRO,-PAPER/NAPKIN,-COMPOSTABLE, 24X24CM, 100-EA/CS", {"pack_qty": 100, "dim_a_mm": 240, "dim_b_mm": 240}),
    ("cocktail napkin compostable 24x24cm *SUB 2842*", {"dim_a_mm": 240, "dim_b_mm": 240}),
    ("Water drystore 6x ITEM", {"pack_qty": 6}),
    ("Ale AMB 24x", {"pack_qty": 24, "storage": "ambient"}),
    ("serenia comb bamboo caseof 100", {"pack_qty": 100}),
    ("FETA-AGED-12 MONTHS REFRIGERATED 1 KGDISC", {"size_qty": 1000, "size_kind": "mass", "storage": "chilled"}),
    ("REF#A19718GOLDLEAF FISH", {}),
    ("SPEC PANTRY PRO 400 MLX2", {"pack_qty": 2, "size_qty": 400, "size_kind": "volume"}),
    ("ROYAL TABLE JCE DRY STORECTN12 NEW", {"pack_qty": 12, "storage": "ambient"}),
    ("HAND-TOWEL, GOLDLEAF,BOX-6", {"pack_qty": 6}),
    ("KNIFE, CHEF'SCHOICE,/20CM NEW", {"dim_a_mm": 200}),
    ("*** USE ITEM 83347 *** GOLDLEAF, BAKING TRAY, PERFORATED, 46X33CM ...", {"dim_a_mm": 460, "dim_b_mm": 330}),
    ("ITEM 93156 FROZEN", {"storage": "frozen"}),
    ("filo astryPantry Pro 10X", {"pack_qty": 10}),
    ("IMP ROYAL TABLE,25CM", {"dim_a_mm": 250}),
    ("PLATE, 27 CM NEW", {"dim_a_mm": 270}),
    ("nife Prime Kitchen 150mm ITEM 27505", {"dim_a_mm": 150}),
    ("OBNING KNIFE CHEF'S CHOICE 100 MM NSF", {"dim_a_mm": 100}),
    # rows from the second review: dimension-only products and an N/1 pack
    ("Chef's Choice knife 20cm", {"dim_a_mm": 200, "size": "200 mm"}),
    ("zip lock-bag/30cm", {"dim_a_mm": 300, "size": "300 mm", "pack_qty": np.nan}),  # not pack 30 from "bag/30"
    ("dinner napikn 40x40cm", {"dim_a_mm": 400, "dim_b_mm": 400, "size": "400 x 400 mm"}),
    ("ROYAL TABLE, 530X320MM", {"dim_a_mm": 530, "dim_b_mm": 320, "size": "530 x 320 mm"}),
    ("cocktail napkin compostable 24x24cm *SUB 2842*", {"dim_a_mm": 240, "dim_b_mm": 240, "size": "240 x 240 mm"}),
    ("skim mlk homogenised 9/1", {"pack_qty": 9, "size": None}),
    ("PROMO greek yog 12/1", {"pack_qty": 12}),
    ("Harbour Select Gin 33", {"pack_qty": np.nan, "size_qty": np.nan}),  # bare number stays unparsed
    ("Pantry Pro 1KG", {"size": "1 kg"}),
    ("Ivory 2.5KG frozen", {"size": "2.5 kg", "storage": "frozen"}),
    ("Chardonnay 750ML", {"size": "750 ml"}),
    ("Coffee 1.5LT", {"size": "1.5 L"}),
    # residual gaps from the full-list pass
    ("PENNNE 5KGDRY STORE NEW", {"size": "5 kg", "storage": "ambient"}),
    ("GOLDLEAF GLVES 1000 PERCASE", {"pack_qty": 1000}),
    ("GoldLeaf spatua 1 doze", {"pack_qty": 12}),
    ("Royal Table Roll/FRZ-2/dozen", {"pack_qty": 24, "storage": "frozen"}),
    ("Chef'sChoice Skinless 4/ct", {"pack_qty": 4}),
    ("METRO VALUE CREAM CHEESE BLOCK CHLD 6/CSREF#A85000", {"pack_qty": 6}),
    ("Metro Value Croissant 48/ PROMO", {"pack_qty": 48}),
    ("Bangkok Thighx6", {"pack_qty": 6}),
    ("Coastal ROYALTABLE 90 X/180/CM", {"dim_a_mm": 900, "dim_b_mm": 1800}),
]


def _matches(got, want) -> bool:
    if isinstance(want, float) and np.isnan(want):
        return got is None or (isinstance(got, float) and np.isnan(got))
    return got == want


review_failures = []
for raw, want in REVIEW_CASES:
    got = parse_description(raw)
    bad = {k: (v, got[k]) for k, v in want.items() if not _matches(got[k], v)}
    if bad:
        review_failures.append((raw, bad))

for raw, bad in review_failures:
    print(raw, "-> (expected, got):", bad)
assert not review_failures, "review cases still failing"
print(f"all {len(REVIEW_CASES)} review cases pass")

## 5. Run on train and measure coverage

`has_pack`, `has_size`, `has_dims` and `storage` are the four fields.
Coverage is shown per supplier, since each supplier formats descriptions differently.

In [ ]:
parsed = parse_frame(train)

print("overall coverage")
print(parsed[["has_pack", "has_size", "has_dims"]].mean().round(3))
print("storage found:", round(parsed["storage"].notna().mean(), 3))
print(parsed["storage"].value_counts(dropna=False))

coverage = (
    pd.concat([train[["supplier"]], parsed[["has_pack", "has_size"]]], axis=1)
    .groupby("supplier")
    .mean()
    .round(2)
)
coverage["rows"] = train["supplier"].value_counts()
coverage

## 6. Residual: rows with no pack and no size

Most residual rows are truncated text (`Iceberg LLettuce 500`, `Chef's Choice Butter 82% fa`),
which no parser can recover. The sample below is for finding *parseable* gaps.

In [ ]:
residual_mask = (
    (parsed["has_pack"] == 0) & (parsed["has_size"] == 0) & (parsed["has_dims"] == 0)
    & train["description"].str.contains(r"\d")
)
print("residual rows:", residual_mask.sum(), "of", len(train))
train.loc[residual_mask, ["supplier", "description"]].sample(40, random_state=3)

## 7. Price per normalised unit

`unit_price` is per selling unit (`unit_of_measure`). For container UOMs (CS, CTN, PK,
BOX) the selling unit holds `pack_qty` items, so total content = `size_qty * pack_qty`.
KG rows are already priced per kg.

In [ ]:
_CONTAINER_UOM = {"CS", "CTN", "PK", "BOX"}


def add_price_features(df: pd.DataFrame, parsed: pd.DataFrame) -> pd.DataFrame:
    uom = df["unit_of_measure"].str.upper()
    price = df["unit_price"].astype(float)

    is_container = uom.isin(_CONTAINER_UOM)
    units_per_sell = parsed["pack_qty"].where(is_container, 1.0).fillna(1.0)
    total = parsed["size_qty"] * units_per_sell  # NaN where no size stated

    kind = parsed["size_kind"]
    out = pd.DataFrame(index=df.index)
    out["price_per_kg"] = np.where(
        uom == "KG", price, np.where(kind == "mass", price / (total / 1000.0), np.nan)
    )
    out["price_per_litre"] = np.where(kind == "volume", price / (total / 1000.0), np.nan)
    out["price_per_item"] = np.where(
        is_container & parsed["pack_qty"].notna(), price / parsed["pack_qty"], price
    )
    out["total_content"] = total
    return out


price_feats = add_price_features(train, parsed)
pd.concat([train[["description", "unit_of_measure", "unit_price"]], parsed[["pack_qty", "size_qty", "size_kind"]], price_feats], axis=1).sample(15, random_state=1)

## 8. Feature matrix

Numeric parsed features plus flags, price features and UOM one-hot. The text features
(char n-gram TF-IDF on `normalise_text(description)`) are added in section 9.

In [ ]:
features = pd.concat(
    [
        parsed[["pack_qty", "size_qty", "dim_a_mm", "dim_b_mm", "has_pack", "has_size", "has_dims"]],
        price_feats[["price_per_kg", "price_per_litre", "price_per_item", "total_content"]],
    ],
    axis=1,
)
features = pd.concat([features, pd.get_dummies(train["unit_of_measure"], prefix="uom", dtype=int)], axis=1)
features["log_unit_price"] = np.log1p(train["unit_price"])
features = features.apply(lambda c: np.log1p(c) if c.name in {"pack_qty", "size_qty", "total_content", "price_per_kg", "price_per_litre", "price_per_item"} else c)
features.describe().T[["count", "mean", "min", "max"]].round(2)

## 9. Encode and scale: the numeric feature matrix

The model-ready matrix. Every feature column is numeric, one-hot encoded or scaled. Only `id`
is not a feature; `category` is the label in the train file and is not a feature either.

- **text:** character n-gram TF-IDF (2 to 5 characters, min_df 2, sublinear tf) on the normalised text.
- **categorical:** one-hot for supplier, unit of measure, buyer group and Metro SKU segment.
- **numeric:** log1p, then standardised. A missing value becomes 0; the `has_*` flags say which.

Fitted on train only; test is transformed with the train fit. The parsed string columns
(`size`, `storage`, `size_kind`) and the raw description are not in the matrix.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

test = pd.read_csv(DATA / "test.csv")


def metro_segment(sku) -> str:
    """Metro SKUs carry a category segment: MHS-KEQ-4180 -> KEQ. Other suppliers -> 'none'."""
    m = re.match(r"MHS-([A-Z]{3})-", str(sku))
    return m.group(1) if m else "none"


# Test rows whose supplier has labelled training rows
covered = test["supplier"].isin(train["supplier"])
print(f"test rows with a labelled same-supplier history: {covered.sum()} of {len(test)}")
print(test.loc[~covered, "supplier"].value_counts().to_string())

PROCESS = DATA / "process"
PROCESS.mkdir(parents=True, exist_ok=True)


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    parsed_df = parse_frame(df)
    price_df = add_price_features(df, parsed_df)
    base = pd.DataFrame(
        {
            "id": df["id"],
            "supplier": df["supplier"],
            "supplier_sku": df["supplier_sku"],
            "description": df["description"],
            "unit_of_measure": df["unit_of_measure"],
            "unit_price": df["unit_price"],
            "buyer_group": df["buyer_group"],
            "text": df["description"].map(normalise_text),
            "metro_segment": df["supplier_sku"].map(metro_segment),
        }
    )
    parsed_cols = ["pack_qty", "size_qty", "size_kind", "dim_a_mm", "dim_b_mm", "storage", "size",
                   "has_pack", "has_size", "has_dims"]
    return pd.concat([base, parsed_df[parsed_cols], price_df], axis=1)


train_out = build_features(train)
train_out["category"] = train["category"]
test_out = build_features(test)

TEXT_COL = "text"
CAT_COLS = ["supplier", "unit_of_measure", "buyer_group", "metro_segment"]
NUM_COLS = ["pack_qty", "size_qty", "dim_a_mm", "dim_b_mm", "has_pack", "has_size", "has_dims",
            "price_per_kg", "price_per_litre", "price_per_item"]


def design_inputs(df: pd.DataFrame) -> pd.DataFrame:
    out = df[[TEXT_COL, *CAT_COLS]].copy()
    out[TEXT_COL] = out[TEXT_COL].fillna("")
    return pd.concat([out, np.log1p(df[NUM_COLS].astype(float)).fillna(0.0)], axis=1)


design = ColumnTransformer(
    [
        ("text", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True), TEXT_COL),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ("num", StandardScaler(), NUM_COLS),
    ],
    sparse_threshold=0,  # dense output, so the CSV is a plain numeric table
)
A_train = design.fit_transform(design_inputs(train_out))  # fitted on train only
A_test = design.transform(design_inputs(test_out))
feature_names = list(design.get_feature_names_out())
assert len(set(feature_names)) == len(feature_names), "duplicate feature names"

train_features = pd.DataFrame(A_train, columns=feature_names)
train_features.insert(0, "id", train_out["id"].values)
train_features["category"] = train_out["category"].values  # label, not a feature
test_features = pd.DataFrame(A_test, columns=feature_names)
test_features.insert(0, "id", test_out["id"].values)

train_features.to_csv(PROCESS / "train_features.csv", index=False, float_format="%.6g")
test_features.to_csv(PROCESS / "test_features.csv", index=False, float_format="%.6g")

reloaded = pd.read_csv(PROCESS / "train_features.csv")
numeric = reloaded.drop(columns=["id", "category"])
assert numeric.apply(lambda c: pd.api.types.is_numeric_dtype(c)).all(), "a feature column is not numeric"
assert not numeric.isna().any().any(), "NaN in the feature matrix"
print("train features:", train_features.shape, "| test features:", test_features.shape)
print("all feature columns numeric, no NaN:", True)
print("first columns:", list(reloaded.columns[:6]), "...", list(reloaded.columns[-3:]))

## 10. Dimensionality reduction: PCA

The design matrix is wide: about 8,900 character n-gram columns plus one-hot and numeric columns.
PCA compresses it into a few uncorrelated components.

- **Fitted on train only.** PCA is unsupervised, so it uses no labels. Test rows are only transformed.
- **Elbow.** The scree plot shows the variance of each component. The elbow is the component furthest
  from the straight line joining the first and last points. The 95% cumulative variance count is shown
  for comparison.
- **Check.** LinearSVC is cross-validated on the first n components, for several n, against the full
  design matrix. This shows whether the elbow choice loses accuracy. It is a sanity check, not tuning.
  PCA was fitted on all train rows before the folds, so these scores are slightly optimistic.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.svm import LinearSVC
import matplotlib.pyplot as plt

# A_train and A_test are the numeric feature matrix from section 9 (dense, train-fitted).
print("design matrix: train", A_train.shape, "| test", A_test.shape)

In [ ]:
MAX_COMPONENTS = 300
pca = PCA(n_components=MAX_COMPONENTS, svd_solver="randomized", random_state=0).fit(A_train)
variance = pca.explained_variance_ratio_
cumulative = np.cumsum(variance)


def elbow(values) -> int:
    """Components up to the bend: the point furthest from the chord joining the first and last points."""
    v = np.asarray(values, dtype=float)
    x = np.linspace(0, 1, len(v))
    v = (v - v.min()) / (v.max() - v.min())
    chord = v[0] + (v[-1] - v[0]) * x
    return int(np.argmax(np.abs(chord - v))) + 1


N_ELBOW = elbow(variance)
N_95 = int(np.searchsorted(cumulative, 0.95)) + 1
print(f"elbow at {N_ELBOW} components ({cumulative[N_ELBOW - 1]:.1%} of variance)")
print(f"95% of variance needs {N_95} components")

k = np.arange(1, MAX_COMPONENTS + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(k, variance, marker=".", linewidth=1)
ax[0].axvline(N_ELBOW, color="tab:red", linestyle="--", label=f"elbow = {N_ELBOW}")
ax[0].set_title("Scree: variance explained by each component")
ax[0].set_xlabel("component")
ax[0].set_ylabel("explained variance ratio")
ax[0].set_xlim(0, 100)
ax[0].legend()
ax[1].plot(k, cumulative, linewidth=1)
ax[1].axhline(0.95, color="grey", linestyle=":", label="95%")
ax[1].axvline(N_ELBOW, color="tab:red", linestyle="--", label=f"elbow = {N_ELBOW}")
ax[1].set_title("Cumulative variance")
ax[1].set_xlabel("components")
ax[1].set_ylabel("cumulative variance ratio")
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sanity check: LinearSVC CV macro-F1 on the first n components, against the full design matrix.
y_train = train_out["category"]
LABELS = sorted(y_train.unique())
cv_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def cv_macro_f1_on(matrix) -> float:
    pred = cross_val_predict(LinearSVC(C=0.5, class_weight="balanced", max_iter=5000),
                             matrix, y_train, cv=cv_check, n_jobs=-1)
    return f1_score(y_train, pred, average="macro", labels=LABELS, zero_division=0)


check_sizes = sorted({N_ELBOW, N_95, 25, 50, 100, 200})
rows = [{"input": "full design matrix", "columns": A_train.shape[1],
         "cv_macro_f1": round(cv_macro_f1_on(A_train), 4)}]
for n in check_sizes:
    rows.append({"input": f"PCA first {n}" + (" (elbow)" if n == N_ELBOW else ""), "columns": n,
                 "cv_macro_f1": round(cv_macro_f1_on(pca.transform(A_train)[:, :n]), 4)})
pd.DataFrame(rows)

In [ ]:
# Save the reduced features at the elbow. Test rows are transformed with the train-fitted PCA.
pc_cols = [f"pc_{i}" for i in range(1, N_ELBOW + 1)]
pca_train = pd.DataFrame(pca.transform(A_train)[:, :N_ELBOW], columns=pc_cols)
pca_train.insert(0, "id", train_out["id"].values)
pca_train["category"] = train_out["category"].values
pca_test = pd.DataFrame(pca.transform(A_test)[:, :N_ELBOW], columns=pc_cols)
pca_test.insert(0, "id", test_out["id"].values)

pca_train.to_csv(PROCESS / "train_pca.csv", index=False)
pca_test.to_csv(PROCESS / "test_pca.csv", index=False)
print("saved", PROCESS / "train_pca.csv", pca_train.shape, "|", PROCESS / "test_pca.csv", pca_test.shape)